In [34]:
from pathlib import Path
import subprocess
from io import StringIO

import numpy as np
import pandas as pd

# =========================================================
# PATHS
# =========================================================
BASE_DIR = Path.cwd()   # current working directory
DATA_DIR = BASE_DIR / "EDS" / "Data" / "Output"

IN_PKL = DATA_DIR / "acs_ssc_predicted_merged.pkl"
OUT_PKL = DATA_DIR / "acs_ssc_msub_merged.pkl"

# =========================================================
# LOAD
# =========================================================
df = pd.read_pickle(IN_PKL)
print(f"Loaded: {df.shape}")

# =========================================================
# FIPS -> TAXSIM SOI state code
# =========================================================
FIPS_TO_SOI = {
    1:1,  2:2,  4:3,  5:4,  6:5,  8:6,  9:7,  10:8,  11:9,  12:10,
    13:11, 15:12, 16:13, 17:14, 18:15, 19:16, 20:17, 21:18, 22:19, 23:20,
    24:21, 25:22, 26:23, 27:24, 28:25, 29:26, 30:27, 31:28, 32:29, 33:30,
    34:31, 35:32, 36:33, 37:34, 38:35, 39:36, 40:37, 41:38, 42:39, 44:40,
    45:41, 46:42, 47:43, 48:44, 49:45, 50:46, 51:47, 53:48, 54:49, 55:50, 56:51
}

# =========================================================
# HELPERS
# =========================================================

import subprocess
import tempfile
import requests
from io import StringIO
def run_taxsim_local(taxsim_input_df: pd.DataFrame) -> pd.DataFrame:
    exe_path = Path.home() / "Desktop" / "taxsim35" / "taxsim35.exe"

    csv_str = taxsim_input_df.to_csv(index=False)

    result = subprocess.run(
        [str(exe_path)],
        input=csv_str,
        capture_output=True,
        text=True
    )

    if result.returncode != 0:
        raise RuntimeError(f"TAXSIM error:\n{result.stderr}")

    return pd.read_csv(StringIO(result.stdout))
def safe_col(frame: pd.DataFrame, col: str, default: float = 0.0) -> pd.Series:
    if col in frame.columns:
        return pd.to_numeric(frame[col], errors="coerce").fillna(default)
    return pd.Series(default, index=frame.index, dtype="float64")


def clip_nonneg(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors="coerce").fillna(0.0).clip(lower=0.0)


def build_observed_components(df: pd.DataFrame) -> pd.DataFrame:
    """
    Build observed TAXSIM inputs from reported ACS income/housing variables.

    Key rule:
    - pwages/swages = wage income only
    - business + investment go to otherprop
    - retirement goes to pensions
    - social security goes to gssi
    - welfare/support go to transfers
    """

    out = df.copy()

    # ----------------------------
    # Core income components
    # ----------------------------
    # Wages only
    out["r_pwages_obs"] = clip_nonneg(safe_col(out, "r_incwage"))
    out["sp_pwages_obs"] = clip_nonneg(safe_col(out, "sp_incwage"))

    # Other property income / non-wage taxable-ish flows
    # Put business + investment + other into otherprop as a practical approximation
    out["r_otherprop_obs"] = (
        clip_nonneg(safe_col(out, "r_incbus00")) +
        clip_nonneg(safe_col(out, "r_incinvst")) +
        clip_nonneg(safe_col(out, "r_incother"))
    )
    out["sp_otherprop_obs"] = (
        clip_nonneg(safe_col(out, "sp_incbus00")) +
        clip_nonneg(safe_col(out, "sp_incinvst")) +
        clip_nonneg(safe_col(out, "sp_incother"))
    )

    # Retirement / pensions
    out["r_pensions_obs"] = clip_nonneg(safe_col(out, "r_incretir"))
    out["sp_pensions_obs"] = clip_nonneg(safe_col(out, "sp_incretir"))

    # Social Security
    out["r_gssi_obs"] = clip_nonneg(safe_col(out, "r_incss"))
    out["sp_gssi_obs"] = clip_nonneg(safe_col(out, "sp_incss"))

    # Transfers
    out["r_transfers_obs"] = (
        clip_nonneg(safe_col(out, "r_incwelfr")) +
        clip_nonneg(safe_col(out, "r_incsupp"))
    )
    out["sp_transfers_obs"] = (
        clip_nonneg(safe_col(out, "sp_incwelfr")) +
        clip_nonneg(safe_col(out, "sp_incsupp"))
    )

    # Long-term capital gains not separately available here
    out["r_ltcg_obs"] = 0.0
    out["sp_ltcg_obs"] = 0.0

    # ----------------------------
    # Housing inputs
    # ----------------------------
    # Rent: ACS rent is monthly, convert to annual
    out["joint_rentpaid_obs"] = clip_nonneg(safe_col(out, "rent")) * 12.0
    out["single_rentpaid_obs"] = out["joint_rentpaid_obs"] / 2.0

    # Property tax from proptx99 interval code
    ptx = clip_nonneg(safe_col(out, "proptx99")).astype(int)
    joint_proptax = pd.Series(0.0, index=out.index, dtype="float64")

    # Rough midpoint approximation
    joint_proptax.loc[ptx <= 1] = 0.0
    joint_proptax.loc[(ptx >= 2) & (ptx <= 21)] = (ptx.loc[(ptx >= 2) & (ptx <= 21)] - 1) * 50.0
    joint_proptax.loc[(ptx >= 22) & (ptx <= 31)] = 1000.0 + (ptx.loc[(ptx >= 22) & (ptx <= 31)] - 22) * 100.0
    joint_proptax.loc[ptx > 31] = 2500.0

    out["joint_proptax_obs"] = joint_proptax.fillna(0.0)
    out["single_proptax_obs"] = out["joint_proptax_obs"] / 2.0

    # Mortgage proxy
    mort1 = clip_nonneg(safe_col(out, "mortamt1"))
    mort2 = clip_nonneg(safe_col(out, "mortamt2"))
    out["joint_mortgage_obs"] = mort1 + mort2
    out["single_mortgage_obs"] = out["joint_mortgage_obs"] / 2.0

    # ----------------------------
    # Dependent counts
    # ----------------------------
    out["depx"] = clip_nonneg(safe_col(out, "c_children")).astype(int)
    out["dep13"] = clip_nonneg(safe_col(out, "c_children0_12")).astype(int)
    out["dep17"] = clip_nonneg(safe_col(out, "c_children0_16")).astype(int)
    out["dep18"] = clip_nonneg(safe_col(out, "c_children0_18")).astype(int)

    return out


def build_predicted_components(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    out["r_pwages_hat"] = clip_nonneg(safe_col(out, "hat_incearn_r"))
    out["sp_pwages_hat"] = clip_nonneg(safe_col(out, "hat_incearn_sp"))

    for c in [
        "r_otherprop_hat", "sp_otherprop_hat",
        "r_pensions_hat", "sp_pensions_hat",
        "r_gssi_hat", "sp_gssi_hat",
        "r_transfers_hat", "sp_transfers_hat",
        "r_ltcg_hat", "sp_ltcg_hat",
        "joint_rentpaid_hat", "single_rentpaid_hat",
        "joint_proptax_hat", "single_proptax_hat",
        "joint_mortgage_hat", "single_mortgage_hat",
    ]:
        out[c] = 0.0

    out["depx"] = clip_nonneg(safe_col(out, "c_children")).astype(int)
    out["dep13"] = clip_nonneg(safe_col(out, "c_children0_12")).astype(int)
    out["dep17"] = clip_nonneg(safe_col(out, "c_children0_16")).astype(int)
    out["dep18"] = clip_nonneg(safe_col(out, "c_children0_18")).astype(int)

    return out


def build_taxsim_rows(df: pd.DataFrame, suffix: str) -> pd.DataFrame:
    """
    Build 3 TAXSIM rows per couple:
    1) higher earner as single
    2) lower earner as single
    3) joint couple

    suffix: "obs" or "hat"
    """
    rows = []

    # column families
    pwages_r = f"r_pwages_{suffix}"
    pwages_sp = f"sp_pwages_{suffix}"

    r_otherprop = f"r_otherprop_{suffix}"
    sp_otherprop = f"sp_otherprop_{suffix}"
    r_pensions = f"r_pensions_{suffix}"
    sp_pensions = f"sp_pensions_{suffix}"
    r_gssi = f"r_gssi_{suffix}"
    sp_gssi = f"sp_gssi_{suffix}"
    r_transfers = f"r_transfers_{suffix}"
    sp_transfers = f"sp_transfers_{suffix}"
    r_ltcg = f"r_ltcg_{suffix}"
    sp_ltcg = f"sp_ltcg_{suffix}"

    joint_rentpaid = f"joint_rentpaid_{suffix}"
    single_rentpaid = f"single_rentpaid_{suffix}"
    joint_proptax = f"joint_proptax_{suffix}"
    single_proptax = f"single_proptax_{suffix}"
    joint_mortgage = f"joint_mortgage_{suffix}"
    single_mortgage = f"single_mortgage_{suffix}"

    for i, row in enumerate(df.itertuples(index=False), start=0):
        year = int(row.year)
        statefip = int(row.statefip)
        soi = FIPS_TO_SOI.get(statefip, 0)

        age_r = int(row.age)
        age_sp = int(row.sp_age)

        # partner-specific amounts
        er = max(float(getattr(row, pwages_r, 0.0) or 0.0), 0.0)
        es = max(float(getattr(row, pwages_sp, 0.0) or 0.0), 0.0)

        rp = max(float(getattr(row, r_otherprop, 0.0) or 0.0), 0.0)
        sp = max(float(getattr(row, sp_otherprop, 0.0) or 0.0), 0.0)

        rpen = max(float(getattr(row, r_pensions, 0.0) or 0.0), 0.0)
        spen = max(float(getattr(row, sp_pensions, 0.0) or 0.0), 0.0)

        rgssi = max(float(getattr(row, r_gssi, 0.0) or 0.0), 0.0)
        sgssi = max(float(getattr(row, sp_gssi, 0.0) or 0.0), 0.0)

        rtrans = max(float(getattr(row, r_transfers, 0.0) or 0.0), 0.0)
        strans = max(float(getattr(row, sp_transfers, 0.0) or 0.0), 0.0)

        rltcg = max(float(getattr(row, r_ltcg, 0.0) or 0.0), 0.0)
        sltcg = max(float(getattr(row, sp_ltcg, 0.0) or 0.0), 0.0)

        depx = int(getattr(row, "depx", 0) or 0)
        dep13 = int(getattr(row, "dep13", 0) or 0)
        dep17 = int(getattr(row, "dep17", 0) or 0)
        dep18 = int(getattr(row, "dep18", 0) or 0)
        

        # ages / higher-lower ordering
        if er >= es:
            hi_earn, lo_earn = er, es
            age_hi, age_lo = age_r, age_sp
            hi_otherprop, lo_otherprop = rp, sp
            hi_pensions, lo_pensions = rpen, spen
            hi_gssi, lo_gssi = rgssi, sgssi
            hi_transfers, lo_transfers = rtrans, strans
            hi_ltcg, lo_ltcg = rltcg, sltcg
        else:
            hi_earn, lo_earn = es, er
            age_hi, age_lo = age_sp, age_r
            hi_otherprop, lo_otherprop = sp, rp
            hi_pensions, lo_pensions = spen, rpen
            hi_gssi, lo_gssi = sgssi, rgssi
            hi_transfers, lo_transfers = strans, rtrans
            hi_ltcg, lo_ltcg = sltcg, rltcg

        # Approximation for single-return dependent assignment:
        # split children as evenly as possible across the two hypothetical single filers
        depx_hi = depx
        depx_lo = 0
        dep13_hi = dep13
        dep13_lo = 0
        dep17_hi = dep17
        dep17_lo = 0
        dep18_hi = dep18
        dep18_lo = 0

        # Higher earner single
        rows.append({
            "taxsimid": i * 3 + 1,
            "year": year,
            "state": soi,
            "mstat": 1,
            "page": age_hi,
            "sage": 0,
            "depx": depx_hi,
            "dep13": dep13_hi,
            "dep17": dep17_hi,
            "dep18": dep18_hi,
            "pwages": hi_earn,
            "swages": 0.0,
            "otherprop": hi_otherprop,
            "pensions": hi_pensions,
            "gssi": hi_gssi,
            "transfers": hi_transfers,
            "ltcg": hi_ltcg,
            "rentpaid": float(getattr(row, single_rentpaid, 0.0) or 0.0),
            "proptax": float(getattr(row, single_proptax, 0.0) or 0.0),
            "mortgage": float(getattr(row, single_mortgage, 0.0) or 0.0),
        })

        # Lower earner single
        rows.append({
            "taxsimid": i * 3 + 2,
            "year": year,
            "state": soi,
            "mstat": 1,
            "page": age_lo,
            "sage": 0,
            "depx": depx_lo,
            "dep13": dep13_lo,
            "dep17": dep17_lo,
            "dep18": dep18_lo,
            "pwages": lo_earn,
            "swages": 0.0,
            "otherprop": lo_otherprop,
            "pensions": lo_pensions,
            "gssi": lo_gssi,
            "transfers": lo_transfers,
            "ltcg": lo_ltcg,
            "rentpaid": float(getattr(row, single_rentpaid, 0.0) or 0.0),
            "proptax": float(getattr(row, single_proptax, 0.0) or 0.0),
            "mortgage": float(getattr(row, single_mortgage, 0.0) or 0.0),
        })

        # Joint return
        rows.append({
            "taxsimid": i * 3 + 3,
            "year": year,
            "state": soi,
            "mstat": 2,
            "page": age_hi,
            "sage": age_lo,
            "depx": depx,
            "dep13": dep13,
            "dep17": dep17,
            "dep18": dep18,
            "pwages": hi_earn,
            "swages": lo_earn,
            "otherprop": hi_otherprop + lo_otherprop,
            "pensions": hi_pensions + lo_pensions,
            "gssi": hi_gssi + lo_gssi,
            "transfers": hi_transfers + lo_transfers,
            "ltcg": hi_ltcg + lo_ltcg,
            "rentpaid": float(getattr(row, joint_rentpaid, 0.0) or 0.0),
            "proptax": float(getattr(row, joint_proptax, 0.0) or 0.0),
            "mortgage": float(getattr(row, joint_mortgage, 0.0) or 0.0),
        })

    out = pd.DataFrame(rows)

    # Ensure required numeric types and nonnegative where needed
    num_cols = [
        "pwages", "swages", "otherprop", "pensions", "gssi", "transfers",
        "ltcg", "rentpaid", "proptax", "mortgage"
    ]
    for c in num_cols:
        out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0.0).clip(lower=0.0)

    int_cols = ["year", "state", "mstat", "page", "sage", "depx", "dep13", "dep17", "dep18", "taxsimid"]
    for c in int_cols:
        out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0).astype(int)

    return out


def compute_subsidies_taxsim(df: pd.DataFrame, suffix: str) -> pd.DataFrame:
    print(f"Building TAXSIM input ({suffix})...")
    inp = build_taxsim_rows(df, suffix=suffix)
    print(f"Sending {len(inp):,} rows to TAXSIM...")

    out = run_taxsim_local(inp)

    expected_rows = len(df) * 3
    if len(out) != expected_rows:
        raise RuntimeError(
            f"TAXSIM returned {len(out)} rows but expected {expected_rows}.\n"
            f"Columns: {out.columns.tolist()}\n"
            f"Head:\n{out.head()}"
        )

    # safer than relying on taxsimid lookup
    out = out.sort_values("taxsimid").reset_index(drop=True)

    fiitax = out["fiitax"].to_numpy()
    siitax = out["siitax"].to_numpy()

    t_hi_fed = fiitax[0::3]
    t_lo_fed = fiitax[1::3]
    t_jt_fed = fiitax[2::3]

    t_hi_st = siitax[0::3]
    t_lo_st = siitax[1::3]
    t_jt_st = siitax[2::3]

    msub_fed = (t_hi_fed + t_lo_fed) - t_jt_fed
    msub_state = (t_hi_st + t_lo_st) - t_jt_st

    no_state_recog = (df["staterecog_policy"].to_numpy() == 0)
    pre_windsor = (df["preW"].to_numpy() == 1)

    msub_state[no_state_recog] = 0.0
    msub_fed[pre_windsor] = 0.0
    msub_state[pre_windsor] = 0.0

    msub_total = msub_fed + msub_state

    out_df = df.copy()
    out_df[f"msub_fed_{suffix}"] = msub_fed
    out_df[f"msub_state_{suffix}"] = msub_state
    out_df[f"msub_total_{suffix}"] = msub_total
    return out_df

def print_diagnostics(df: pd.DataFrame) -> None:
    print("\n" + "=" * 70)
    print("MERGED TAXSIM DIAGNOSTICS")
    print("=" * 70)

    for label, mask in [
        ("Married", df["sscouple_mar"] == 1),
        ("Cohabiting", df["sscouple_coh"] == 1),
    ]:
        sub = df.loc[mask]
        print(f"\n{label} (N={len(sub):,}):")
        print(f"  Fed+state observed:  {sub['msub_total_obs'].mean():8.1f}  (paper: mar=442  coh=264)")
        print(f"  Fed+state predicted: {sub['msub_total_hat'].mean():8.1f}  (paper: mar=68   coh=257)")
        print(f"  Federal observed:    {sub['msub_fed_obs'].mean():8.1f}  (paper: mar=395  coh=232)")
        print(f"  State observed:      {sub['msub_state_obs'].mean():8.1f}  (paper: mar=47   coh=32)")




Loaded: (38279, 151)


In [35]:
df = build_observed_components(df)
df = build_predicted_components(df)
check_cols = [
    "r_pwages_obs", "sp_pwages_obs",
    "r_otherprop_obs", "sp_otherprop_obs",
    "r_pensions_obs", "sp_pensions_obs",
    "r_gssi_obs", "sp_gssi_obs",
    "r_transfers_obs", "sp_transfers_obs",
]
print(df[check_cols].describe())

        r_pwages_obs  sp_pwages_obs  r_otherprop_obs  sp_otherprop_obs  \
count   38279.000000   38279.000000     38279.000000      38279.000000   
mean    76832.441652   29770.311137      6875.821678       3843.784451   
std     84603.801048   37058.468179     34220.879249      18271.551258   
min         0.000000       0.000000         0.000000          0.000000   
25%     30000.000000       0.000000         0.000000          0.000000   
50%     55000.000000   21000.000000         0.000000          0.000000   
75%     93000.000000   45000.000000         0.000000          0.000000   
max    736000.000000  660000.000000    666000.000000     441000.000000   

       r_pensions_obs  sp_pensions_obs    r_gssi_obs   sp_gssi_obs  \
count    38279.000000     38279.000000  38279.000000  38279.000000   
mean       504.154863       904.480734    153.775699    499.858147   
std       5155.612923      7113.664368   1608.300945   2825.520034   
min          0.000000         0.000000      0.000000 

In [36]:
test = df.head(20).copy()
test = build_observed_components(test)
test = build_predicted_components(test)
test = compute_subsidies_taxsim(test, suffix="obs")

Building TAXSIM input (obs)...
Sending 60 rows to TAXSIM...


In [37]:
# =========================================================
# RUN
# =========================================================
# Build richer components
df = build_observed_components(df)
df = build_predicted_components(df)

# Observed subsidy
print("Observed subsidy (reported inputs)...")
df = compute_subsidies_taxsim(df, suffix="obs")


# Predicted subsidy
print("Predicted subsidy (LASSO earnings)...")
df = compute_subsidies_taxsim(df, suffix="hat")

# Diagnostics
print_diagnostics(df)

# Save
df.to_pickle(OUT_PKL)
print(f"\nSaved to {OUT_PKL}")

Observed subsidy (reported inputs)...
Building TAXSIM input (obs)...
Sending 114,837 rows to TAXSIM...
Predicted subsidy (LASSO earnings)...
Building TAXSIM input (hat)...
Sending 114,837 rows to TAXSIM...

MERGED TAXSIM DIAGNOSTICS

Married (N=16,147):
  Fed+state observed:     174.7  (paper: mar=442  coh=264)
  Fed+state predicted:    100.7  (paper: mar=68   coh=257)
  Federal observed:       156.7  (paper: mar=395  coh=232)
  State observed:          18.0  (paper: mar=47   coh=32)

Cohabiting (N=22,132):
  Fed+state observed:     127.9  (paper: mar=442  coh=264)
  Fed+state predicted:    280.0  (paper: mar=68   coh=257)
  Federal observed:       109.5  (paper: mar=395  coh=232)
  State observed:          18.5  (paper: mar=47   coh=32)

Saved to /Users/guypigott/python-venv-demo/EDS/Data/Output/acs_ssc_msub_merged.pkl
